# 05 · Régua de ratings e backtest da régua

**Etapas 7 e 7b do pipeline.** Transforma o score calibrado em ratings (faixas com PD ou LGD fixa) e testa a régua safra a safra e na carteira.

## Etapa 7 — a régua

| passo | onde roda | o que faz |
|---|---|---|
| 1 | Spark → tabela pequena | corta o score do DES em 400 faixas finas e agrega contratos, alvo e score por faixa e partição (uma passada) |
| 2 | pandas | junta as faixas finas em N faixas de mesmo volume (10, ou automático) |
| 3 | pandas | fusões, registradas uma a uma: **inversão** encontrada no teste (sempre fundida); opcionalmente vizinhas **indistinguíveis**; faixas abaixo do **piso** de 2% |
| 4 | pandas | régua final: cortes, PD (ou LGD) do rating = taxa observada no DES, intervalo de confiança |
| 5 | Spark | aplica a régua em todos os contratos (`rating`, `valor_regua`) |

**Exemplo.** 10 faixas de 200.000 contratos no DES. No teste, a faixa 4 teve 3,2% e a faixa 3 teve 3,3%: inversão (a faixa de score pior teve menos maus). As duas são fundidas, e a nova faixa C tem PD = (maus das duas no DES) / 400.000 = 3,0%. Resultado: 9 ratings.

Por que as faixas finas: com elas, qualquer número de faixas (inclusive o modo automático, que testa de 15 a 3) é montado em pandas, sem voltar ao Spark.

## Etapa 7b — backtest da régua

Quatro tabelas, todas de uma agregação Spark com **somas** (que podem ser reagregadas em qualquer nível):

| | por safra | por período (partição inteira) |
|---|---|---|
| **por rating** | A: rating × safra | B: rating × partição |
| **total** | C: total × safra, com ponte | D: total × partição, com ponte |

Status de cada linha: **ok** (nenhuma condição), **atenção** (uma), **alerta** (as duas):

- estatística: a PD da régua fica fora do intervalo de 95% da observada;
- materialidade: |observada / régua − 1| ≥ 10%.

Três alertas seguidos de um rating marcam `recalibrar`. A ordem entre ratings vizinhos é testada safra a safra.

In [ ]:
%run ./00_config

In [ ]:
import string

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

N_FAIXAS_FINAS = 400
ROTULOS_RATING = list(string.ascii_uppercase)
PARTICOES_DES = ("treino", "teste")
CORES_STATUS = {"ok": "#0ca30c", "atencao": "#fab219", "alerta": "#d03b3b"}
MARCADOR_STATUS = {"ok": "o", "atencao": "^", "alerta": "X"}
NOME_STATUS = {"ok": "ok", "atencao": "atenção", "alerta": "alerta"}

## 1. Faixas finas e agregação

O score calibrado do DES (treino + teste) é cortado em 400 quantis (`approxQuantile`). Cada faixa fina recebe, por partição: contratos, soma do alvo, soma do alvo ao quadrado (para a variância na LGD) e soma do score.

In [ ]:
def faixas_finas(base, cfg, coluna_score="score_calibrado"):
    """
    Devolve (cortes_finos, tabela) — tabela pandas com uma linha por
    (faixa_fina, particao): n, soma, soma2, soma_score.

    Faixa fina i = score em [corte_{i−1}, corte_i): o corte inferior entra na
    faixa, o superior não. A primeira começa em −infinito e a última vai até
    +infinito.

    Conceito técnico:
        O Bucketizer (pyspark.ml) faz o corte em faixas de forma nativa. Uma
        expressão com 400 condições encadeadas estouraria a pilha do Spark.
    """
    from pyspark.ml.feature import Bucketizer
    alvo = cfg["colunas"]["alvo"]
    des = base.filter(F.col("particao").isin(*PARTICOES_DES))
    cortes = sorted(set(des.approxQuantile(coluna_score, [k / N_FAIXAS_FINAS for k in range(1, N_FAIXAS_FINAS)], 1e-4)))
    cortador = Bucketizer(splits=[-float("inf")] + cortes + [float("inf")],
                          inputCol=coluna_score, outputCol="faixa_fina", handleInvalid="keep")
    y = F.col(alvo).cast("double")
    tabela = (cortador.transform(base.filter(F.col("particao").isin("treino", "teste", "oot")))
              .groupBy(F.col("faixa_fina").cast("int").alias("faixa_fina"), "particao")
              .agg(F.count(F.lit(1)).alias("n"), F.sum(y).alias("soma"), F.sum(y * y).alias("soma2"),
                   F.sum(F.col(coluna_score)).alias("soma_score"))
              .toPandas())
    return cortes, tabela


def _estatisticas(tabela_finas, indices, particoes):
    """Soma n, soma, soma2 e soma_score das faixas finas `indices` nas `particoes`."""
    t = tabela_finas[tabela_finas["faixa_fina"].isin(indices) & tabela_finas["particao"].isin(particoes)]
    return {c: float(t[c].sum()) for c in ("n", "soma", "soma2", "soma_score")}


def _media(e):
    return e["soma"] / e["n"] if e["n"] else np.nan

## 2. Faixas iniciais de mesmo volume

As faixas finas são juntadas em N faixas com volume do DES o mais parecido possível: cada faixa fina vai para a faixa inicial do seu ponto médio de volume acumulado.

In [ ]:
def agrupa_em_n_faixas(tabela_finas, n_faixas):
    """Lista de grupos (listas de faixas finas contíguas) com volume do DES parecido."""
    des = (tabela_finas[tabela_finas["particao"].isin(PARTICOES_DES)]
           .groupby("faixa_fina")["n"].sum().sort_index())
    participacao = des / des.sum()
    ponto_medio = participacao.cumsum() - participacao / 2
    grupo = np.minimum((ponto_medio * n_faixas).astype(int), n_faixas - 1)
    return [list(indices) for _, indices in des.index.to_series().groupby(grupo.values)]

## 3. Fusões

Testes entre faixas vizinhas, sempre **no teste**:

- **PD:** qui-quadrado na tabela 2 × 2 (maus e bons nas duas faixas), com correção de continuidade;
- **LGD:** teste t de Welch (compara duas médias sem supor variâncias iguais), calculado só com contagem, média e variância de cada faixa.

Regras, aplicadas em ciclo até nada mudar:

1. **Inversão:** faixa de score maior com taxa (ou LGD) menor no teste → **sempre funde** (decisão do desenho). Se o teste der p ≤ 5%, a inversão é **significativa** e fica registrada como achado: o modelo não ordena bem naquela região do score.
2. **Indistinguíveis** (se `fundir_indistinguiveis = True`): vizinhas em ordem, mas com p > 5% → funde o par de maior p.
3. **Piso:** faixa com menos de 2% do volume do DES → funde com a vizinha de taxa mais parecida.

Cada fusão vira uma linha no registro: ordem, faixas, motivo, p-valor, significativa.

In [ ]:
def p_valor_vizinhas(e1, e2, tarefa):
    """p-valor do teste de igualdade entre duas faixas (estatísticas do teste)."""
    from scipy.stats import chi2_contingency, ttest_ind_from_stats
    if e1["n"] < 2 or e2["n"] < 2:
        return 1.0
    if tarefa == "pd":
        tabela = [[e1["soma"], e1["n"] - e1["soma"]], [e2["soma"], e2["n"] - e2["soma"]]]
        if min(sum(linha) for linha in tabela) == 0 or e1["soma"] + e2["soma"] == 0 \
                or (e1["n"] - e1["soma"]) + (e2["n"] - e2["soma"]) == 0:
            return 1.0
        return float(chi2_contingency(tabela, correction=True)[1])
    def desvio(e):
        variancia = (e["soma2"] - e["soma"] ** 2 / e["n"]) / (e["n"] - 1)
        return float(np.sqrt(max(variancia, 0.0)))
    return float(ttest_ind_from_stats(_media(e1), desvio(e1), e1["n"], _media(e2), desvio(e2), e2["n"],
                                      equal_var=False).pvalue)


def funde_faixas(grupos, tabela_finas, cfg):
    """
    Aplica as fusões e devolve (grupos_finais, registro_de_fusoes).
    `grupos` é a lista de faixas iniciais (cada uma, uma lista de faixas finas).
    """
    r = cfg["regua"]
    tarefa, nivel = cfg["tarefa"], r["nivel_fusao"]
    grupos = [list(g) for g in grupos]
    registro = []
    total_des = sum(_estatisticas(tabela_finas, g, PARTICOES_DES)["n"] for g in grupos)

    def funde(i, motivo, p_valor):
        rotulo = f"{_rotulo_grupo(grupos[i])} + {_rotulo_grupo(grupos[i + 1])}"
        grupos[i] = grupos[i] + grupos.pop(i + 1)
        registro.append({"ordem": len(registro) + 1, "faixas": rotulo, "motivo": motivo,
                         "p_valor": p_valor, "significativa": bool(p_valor <= nivel)})

    mudou = True
    while mudou and len(grupos) > 1:
        mudou = False
        teste = [_estatisticas(tabela_finas, g, ["teste"]) for g in grupos]
        # 1. inversões no teste: funde a de menor diferença primeiro
        inversoes = [(i, _media(teste[i]) - _media(teste[i + 1])) for i in range(len(grupos) - 1)
                     if _media(teste[i + 1]) < _media(teste[i])]
        if inversoes:
            i = min(inversoes, key=lambda par: par[1])[0]
            funde(i, "inversão no teste", p_valor_vizinhas(teste[i], teste[i + 1], tarefa))
            mudou = True
            continue
        # 2. vizinhas indistinguíveis (opcional): funde o par de maior p-valor
        if r["fundir_indistinguiveis"]:
            p_valores = [(i, p_valor_vizinhas(teste[i], teste[i + 1], tarefa)) for i in range(len(grupos) - 1)]
            i, p = max(p_valores, key=lambda par: par[1])
            if p > nivel:
                funde(i, "indistinguíveis no teste", p)
                mudou = True
                continue
        # 3. piso de representatividade no DES
        des = [_estatisticas(tabela_finas, g, PARTICOES_DES) for g in grupos]
        pequenas = [i for i, e in enumerate(des) if e["n"] / total_des < r["piso_representatividade"]]
        if pequenas:
            i = min(pequenas, key=lambda k: des[k]["n"])
            vizinhas = [j for j in (i - 1, i + 1) if 0 <= j < len(grupos)]
            j = min(vizinhas, key=lambda k: abs(_media(des[k]) - _media(des[i])))
            esquerda = min(i, j)
            funde(esquerda, f"piso de {r['piso_representatividade']:.0%} do DES",
                  p_valor_vizinhas(teste[esquerda], teste[esquerda + 1], tarefa))
            mudou = True
    return grupos, pd.DataFrame(registro, columns=["ordem", "faixas", "motivo", "p_valor", "significativa"])


def _rotulo_grupo(grupo):
    """Rótulo legível de um grupo pelas faixas iniciais que contém (definido em escolhe_faixas)."""
    return "/".join(str(i) for i in sorted({_FAIXA_INICIAL.get(f, f) for f in grupo}))


_FAIXA_INICIAL = {}   # faixa fina → nº da faixa inicial (1, 2, ...), preenchido por escolhe_faixas

## 4. Número de faixas: fixo ou automático

- **Fixo** (`n_faixas = 10`): 10 faixas iniciais, depois as fusões.
- **Automático** (`n_faixas = "auto"`): testa de 15 a 3 faixas e fica com a maior quantidade que, **antes** das fusões, já não tem inversão no teste e tem todas as faixas acima do piso. Se nenhuma quantidade cumprir, usa 10 com fusões.

In [ ]:
def escolhe_faixas(tabela_finas, cfg):
    """Devolve (n_inicial, grupos_iniciais)."""
    global _FAIXA_INICIAL
    r = cfg["regua"]
    n_inicial = r["n_faixas"]
    if n_inicial == "auto":
        n_inicial = 10
        for n in range(r["n_faixas_max"], r["n_faixas_min"] - 1, -1):
            grupos = agrupa_em_n_faixas(tabela_finas, n)
            teste = [_media(_estatisticas(tabela_finas, g, ["teste"])) for g in grupos]
            des = [_estatisticas(tabela_finas, g, PARTICOES_DES)["n"] for g in grupos]
            if all(np.diff(teste) >= 0) and min(des) / sum(des) >= r["piso_representatividade"]:
                n_inicial = n
                break
        print(f"[régua] modo automático: {n_inicial} faixas iniciais")
    grupos = agrupa_em_n_faixas(tabela_finas, n_inicial)
    _FAIXA_INICIAL = {fina: numero + 1 for numero, g in enumerate(grupos) for fina in g}
    return n_inicial, grupos

## 5. Régua final e aplicação

Para cada rating (A = menor risco):

- **cortes:** o rating vai do seu corte inferior (inclusive) ao corte superior (exclusive): score ≥ corte_inferior e score < corte_superior. O primeiro começa em −∞ e o último termina em +∞;
- **valor da régua:** taxa de maus (PD) ou LGD média observada no **DES** (treino + teste);
- **intervalo de 95%:** Jeffreys na PD (a partir de maus e contratos); média ± t · desvio / √n na LGD;
- volumes e valores observados no teste e no OOT, para leitura.

In [ ]:
def intervalo(e, tarefa, nivel=0.95):
    """Intervalo de confiança do valor observado (PD: Jeffreys; LGD: t)."""
    from scipy.stats import beta, t
    a = (1 - nivel) / 2
    if e["n"] == 0:
        return np.nan, np.nan
    if tarefa == "pd":
        return (float(beta.ppf(a, e["soma"] + 0.5, e["n"] - e["soma"] + 0.5)),
                float(beta.ppf(1 - a, e["soma"] + 0.5, e["n"] - e["soma"] + 0.5)))
    if e["n"] < 2:
        return np.nan, np.nan
    media = _media(e)
    desvio = np.sqrt(max((e["soma2"] - e["soma"] ** 2 / e["n"]) / (e["n"] - 1), 0.0))
    meia_largura = float(t.ppf(1 - a, e["n"] - 1) * desvio / np.sqrt(e["n"]))
    return media - meia_largura, media + meia_largura


def monta_regua(grupos, cortes_finos, tabela_finas, cfg):
    """Tabela da régua (pandas), uma linha por rating."""
    linhas, total_des = [], sum(_estatisticas(tabela_finas, g, PARTICOES_DES)["n"] for g in grupos)
    for posicao, grupo in enumerate(grupos):
        des = _estatisticas(tabela_finas, grupo, PARTICOES_DES)
        teste = _estatisticas(tabela_finas, grupo, ["teste"])
        oot = _estatisticas(tabela_finas, grupo, ["oot"])
        superior = max(grupo)
        ic_inf, ic_sup = intervalo(des, cfg["tarefa"])
        linhas.append({
            "rating": ROTULOS_RATING[posicao],
            "faixas_iniciais": _rotulo_grupo(grupo),
            "corte_superior": float(cortes_finos[superior]) if superior < len(cortes_finos) else None,
            "n_des": int(des["n"]), "participacao_des": des["n"] / total_des,
            "valor_regua": _media(des), "ic_inf": ic_inf, "ic_sup": ic_sup,
            "score_medio_des": des["soma_score"] / des["n"] if des["n"] else np.nan,
            "n_teste": int(teste["n"]), "observado_teste": _media(teste),
            "n_oot": int(oot["n"]), "observado_oot": _media(oot),
        })
    regua = pd.DataFrame(linhas)
    regua["corte_inferior"] = [None] + regua["corte_superior"].tolist()[:-1]
    return regua[["rating", "faixas_iniciais", "corte_inferior", "corte_superior"]
                 + [c for c in regua.columns if c not in ("rating", "faixas_iniciais", "corte_inferior", "corte_superior")]]


def aplica_regua(sdf, regua, coluna_score="score_calibrado"):
    """
    Cria `rating` e `valor_regua` (PD ou LGD do rating) a partir dos cortes.
    Convenção de borda: score < corte_superior pertence ao rating (a mesma das faixas finas).
    """
    rating = F.lit(regua["rating"].iloc[-1])
    valor = F.lit(float(regua["valor_regua"].iloc[-1]))
    for _, linha in regua.iloc[:-1][::-1].iterrows():
        condicao = F.col(coluna_score) < F.lit(linha["corte_superior"])
        rating = F.when(condicao, F.lit(linha["rating"])).otherwise(rating)
        valor = F.when(condicao, F.lit(float(linha["valor_regua"]))).otherwise(valor)
    return sdf.withColumn("rating", rating).withColumn("valor_regua", valor)

## 6. Gráfico da fusão

As faixas iniciais no eixo x, com a taxa (ou LGD média) do DES e do teste. Faixas que foram fundidas no mesmo rating aparecem no mesmo bloco de fundo, com o rótulo do rating no topo. As fusões por inversão significativa ficam destacadas em vermelho na tabela de registro.

In [ ]:
def grafico_fusao(n_inicial, grupos_iniciais, regua, tabela_finas, cfg):
    """Taxa por faixa inicial (DES e teste) e os blocos de cada rating final."""
    import matplotlib.pyplot as plt
    des = [_media(_estatisticas(tabela_finas, g, PARTICOES_DES)) for g in grupos_iniciais]
    teste = [_media(_estatisticas(tabela_finas, g, ["teste"])) for g in grupos_iniciais]
    x = np.arange(1, n_inicial + 1)
    figura, eixo = plt.subplots(figsize=(max(7, 0.7 * n_inicial + 2), 4.2), dpi=130)
    inicio = 1
    for posicao, linha in regua.iterrows():
        tamanho = len(str(linha["faixas_iniciais"]).split("/"))
        if posicao % 2 == 0:
            eixo.axvspan(inicio - 0.5, inicio + tamanho - 0.5, color="#f0efec", lw=0, zorder=0)
        eixo.text(inicio + (tamanho - 1) / 2, 1.01, linha["rating"], transform=eixo.get_xaxis_transform(),
                  ha="center", va="bottom", fontsize=9, fontweight="bold", color="#52514e")
        inicio += tamanho
    escala = 100 if cfg["tarefa"] == "pd" else 100
    eixo.plot(x, np.array(des) * escala, marker="o", color="#2a78d6", lw=2, label="DES (treino + teste)")
    eixo.plot(x, np.array(teste) * escala, marker="s", color="#eb6834", lw=1.5, ls=(0, (4, 2)), label="teste")
    eixo.set_xticks(x)
    eixo.set_xlabel("faixa inicial (ordem crescente de score)")
    eixo.set_ylabel("taxa de inadimplência (%)" if cfg["tarefa"] == "pd" else "LGD média (%)")
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(axis="y", color="#e4e3df", lw=0.8)
    eixo.legend(frameon=False, fontsize=8, loc="upper left")
    eixo.set_title(f"Régua: {n_inicial} faixas iniciais → {len(regua)} ratings", loc="left", fontsize=11, pad=18)
    figura.tight_layout()
    return figura

## 7. Backtest da régua (7b): agregação e indicadores

Uma agregação Spark por rating × safra × partição (treino, teste, OOT) com somas: contratos, soma do alvo, soma do alvo ao quadrado, soma do score calibrado e soma do valor da régua. As quatro tabelas saem dessa tabela pequena.

| coluna | significado |
|---|---|
| `observado` | taxa de maus (PD) ou LGD média observada |
| `regua` | valor da régua; no total, a média ponderada pelo mix de ratings |
| `modelo_medio` | média do score calibrado dos contratos (a "PD média do modelo") |
| `ic_inf`, `ic_sup` | intervalo de 95% do observado |
| `desvio_relativo` | observado / régua − 1 |
| `gap_composicao` | modelo_medio − régua: clientes mais (ou menos) arriscados dentro das faixas |
| `gap_calibracao` | observado − modelo_medio: o que o próprio modelo errou |
| `efeito_mix` (total) | régua ponderada − valor do DES: migração entre ratings |

**Ponte da carteira:** valor do DES + efeito mix + gap de composição + gap de calibração = observado.

In [ ]:
def agrega_backtest(base_com_rating, cfg, coluna_score="score_calibrado"):
    """Uma linha por rating × safra × partição, com somas."""
    colunas = cfg["colunas"]
    y = F.col(colunas["alvo"]).cast("double")
    return (base_com_rating.filter(F.col("particao").isin("treino", "teste", "oot"))
            .groupBy("rating", F.col(colunas["safra"]).alias("safra"), "particao")
            .agg(F.count(F.lit(1)).alias("n"), F.sum(y).alias("soma"), F.sum(y * y).alias("soma2"),
                 F.sum(F.col(coluna_score)).alias("soma_score"), F.sum("valor_regua").alias("soma_regua"))
            .toPandas())


def indicadores(tabela, cfg):
    """Observado, régua, modelo médio, intervalo, desvio, status e gaps de cada linha."""
    b = cfg["backtest_regua"]
    t = tabela.copy()
    t["observado"] = t["soma"] / t["n"]
    t["regua"] = t["soma_regua"] / t["n"]
    t["modelo_medio"] = t["soma_score"] / t["n"]
    limites = [intervalo({"n": r.n, "soma": r.soma, "soma2": r.soma2}, cfg["tarefa"], b["nivel"])
               for r in t.itertuples()]
    t["ic_inf"], t["ic_sup"] = [l[0] for l in limites], [l[1] for l in limites]
    t["regua_no_ic"] = (t["regua"] >= t["ic_inf"]) & (t["regua"] <= t["ic_sup"])
    t["desvio_relativo"] = t["observado"] / t["regua"] - 1
    fora = ~t["regua_no_ic"]
    material = t["desvio_relativo"].abs() >= b["tol_relativa"]
    t["status"] = np.select([fora & material, fora | material], ["alerta", "atencao"], default="ok")
    t["gap_composicao"] = t["modelo_medio"] - t["regua"]
    t["gap_calibracao"] = t["observado"] - t["modelo_medio"]
    return t


def marca_persistencia(tabela, n_persistencia):
    """Conta alertas seguidos por rating, na ordem das safras (zera fora do alerta)."""
    t = tabela.sort_values(["rating", "safra"]).copy()
    em_alerta = t["status"].eq("alerta").astype(int)
    blocos = (em_alerta != em_alerta.groupby(t["rating"]).shift()).cumsum()
    t["alertas_seguidos"] = em_alerta.groupby([t["rating"], blocos]).cumsum()
    t["recalibrar"] = t["alertas_seguidos"] >= n_persistencia
    return t


def ordem_por_safra(tabela_a, cfg):
    """
    Inversões entre ratings vizinhos em cada safra: o rating de maior risco
    com observado menor que o anterior. Testa a significância (qui-quadrado
    na PD; Welch na LGD).
    """
    linhas = []
    nivel = cfg["regua"]["nivel_fusao"]
    for (safra, particao), grupo in tabela_a.groupby(["safra", "particao"]):
        g = grupo.sort_values("rating")
        pares, significativas = [], 0
        registros = g.to_dict("records")
        for anterior, seguinte in zip(registros[:-1], registros[1:]):
            if seguinte["observado"] < anterior["observado"]:
                p = p_valor_vizinhas(anterior, seguinte, cfg["tarefa"])
                significativas += int(p <= nivel)
                pares.append(f"{anterior['rating']}>{seguinte['rating']} (p={p:.3f})")
        linhas.append({"safra": safra, "particao": particao, "n_pares": max(len(g) - 1, 0),
                       "n_inversoes": len(pares), "n_inversoes_significativas": significativas,
                       "pares_invertidos": "; ".join(pares)})
    return pd.DataFrame(linhas).sort_values("safra").reset_index(drop=True)


def tabelas_backtest(agregado, regua, cfg):
    """As quatro tabelas do backtest + a tabela de ordem por safra."""
    somas = ["n", "soma", "soma2", "soma_score", "soma_regua"]
    b = cfg["backtest_regua"]
    valor_des = float((regua["valor_regua"] * regua["n_des"]).sum() / regua["n_des"].sum())

    tabela_a = marca_persistencia(indicadores(agregado, cfg), b["n_safras_persistencia"])
    tabela_a = tabela_a.merge(regua[["rating", "corte_inferior", "corte_superior"]], on="rating", how="left")
    tabela_b = indicadores(agregado.groupby(["rating", "particao"], as_index=False)[somas].sum(), cfg)
    tabela_c = indicadores(agregado.groupby(["safra", "particao"], as_index=False)[somas].sum(), cfg)
    tabela_d = indicadores(agregado.groupby("particao", as_index=False)[somas].sum(), cfg)
    for tabela in (tabela_c, tabela_d):
        tabela["valor_des"] = valor_des
        tabela["efeito_mix"] = tabela["regua"] - valor_des
    ordem = ordem_por_safra(tabela_a, cfg)
    ordem_particao = {"treino": 0, "teste": 1, "oot": 2}
    tabela_b = tabela_b.sort_values(["rating", "particao"], key=lambda s: s.map(ordem_particao) if s.name == "particao" else s)
    tabela_d = tabela_d.sort_values("particao", key=lambda s: s.map(ordem_particao))
    return {"rating_safra": tabela_a.reset_index(drop=True), "rating_periodo": tabela_b.reset_index(drop=True),
            "total_safra": tabela_c.sort_values("safra").reset_index(drop=True),
            "total_periodo": tabela_d.reset_index(drop=True), "ordem_por_safra": ordem}

## 8. Gráficos do backtest

1. **Um gráfico por rating:** observado com o intervalo de 95%, régua, média do modelo, status de cada safra (cor e forma) e o fundo marcando treino, teste e OOT.
2. **Ordem por safra:** todos os ratings no mesmo eixo, em escala logarítmica (para o A e o último rating caberem juntos). Linhas que se cruzam = inversão naquela safra.
3. **Mapa de status:** rating × safra, colorido por ok/atenção/alerta.
4. **Ponte da carteira:** do valor do DES ao observado em cada período, em degraus (mix, composição, calibração).

In [ ]:
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
CORES_PARTICAO = {"treino": "#f0efec", "teste": "#e4e3df", "oot": "#fcfcfb"}


def _rotulo_safra(valor):
    data = pd.Timestamp(valor)
    return f"{MESES[data.month - 1]}/{data.year % 100:02d}"


def _fundo_particoes(eixo, particoes):
    """Pinta o fundo de cada bloco contíguo de partição e escreve o nome no topo."""
    inicio = 0
    for i in range(1, len(particoes) + 1):
        if i == len(particoes) or particoes[i] != particoes[inicio]:
            eixo.axvspan(inicio - 0.5, i - 0.5, color=CORES_PARTICAO.get(particoes[inicio], "#fcfcfb"), lw=0, zorder=0)
            eixo.text((inicio + i - 1) / 2, 1.02, f"{particoes[inicio].upper()} ({i - inicio})",
                      transform=eixo.get_xaxis_transform(), ha="center", fontsize=8,
                      color="#52514e", fontweight="bold")
            if inicio > 0:
                eixo.axvline(inicio - 0.5, color="#52514e", lw=1, ls=(0, (4, 3)))
            inicio = i


def grafico_rating_safra(tabela_a, rating, cfg):
    """Gráfico de um rating ao longo das safras."""
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    d = tabela_a[tabela_a["rating"] == rating].sort_values("safra").reset_index(drop=True)
    x = np.arange(len(d))
    figura, eixo = plt.subplots(figsize=(11, 4.4), dpi=130)
    _fundo_particoes(eixo, d["particao"].tolist())
    eixo.fill_between(x, d["ic_inf"] * 100, d["ic_sup"] * 100, color="#cde2fb", lw=0, zorder=2)
    eixo.plot(x, d["regua"] * 100, color="#52514e", lw=1.5, zorder=3)
    eixo.plot(x, d["modelo_medio"] * 100, color="#eb6834", lw=2, ls=(0, (5, 3)), zorder=4)
    eixo.plot(x, d["observado"] * 100, color="#2a78d6", lw=2, zorder=5)
    for status, marcador in MARCADOR_STATUS.items():
        m = d["status"] == status
        eixo.scatter(x[m], d.loc[m, "observado"] * 100, marker=marcador, s=55, color=CORES_STATUS[status],
                     edgecolor="white", linewidth=1.1, zorder=6)
    eixo.text(len(d) - 0.45, d["regua"].iloc[-1] * 100, f" régua {d['regua'].iloc[-1]:.1%}".replace(".", ","),
              va="center", fontsize=8, color="#52514e")
    eixo.set_xticks(x)
    eixo.set_xticklabels([_rotulo_safra(s) for s in d["safra"]], rotation=45, ha="right", fontsize=7)
    eixo.set_xlim(-0.5, len(d) - 0.5)
    eixo.set_ylabel("PD (%)" if cfg["tarefa"] == "pd" else "LGD (%)")
    eixo.grid(axis="y", color="#e4e3df", lw=0.8, zorder=0)
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.set_title(f"Rating {rating}: observado × predito por safra", loc="left", fontsize=11, pad=20)
    legenda = ([Line2D([], [], color="#2a78d6", lw=2, label="observado"), Patch(color="#cde2fb", label="IC 95% do observado"),
                Line2D([], [], color="#52514e", lw=1.5, label="régua"),
                Line2D([], [], color="#eb6834", lw=2, ls=(0, (5, 3)), label="média do modelo")]
               + [Line2D([], [], marker=MARCADOR_STATUS[s], ls="", color=CORES_STATUS[s], markersize=7,
                         markeredgecolor="white", label=NOME_STATUS[s]) for s in MARCADOR_STATUS])
    eixo.legend(handles=legenda, loc="upper left", bbox_to_anchor=(1.07, 1.0), frameon=False, fontsize=8)
    figura.tight_layout()
    return figura


def grafico_ordem_por_safra(tabela_a, cfg):
    """Todos os ratings por safra, escala logarítmica."""
    import matplotlib.pyplot as plt
    safras = sorted(tabela_a["safra"].unique())
    posicao = {s: i for i, s in enumerate(safras)}
    particoes = (tabela_a.drop_duplicates("safra").set_index("safra").loc[safras, "particao"].tolist())
    figura, eixo = plt.subplots(figsize=(11, 5), dpi=130)
    _fundo_particoes(eixo, particoes)
    ratings = sorted(tabela_a["rating"].unique())
    cores = plt.cm.Blues(np.linspace(0.35, 1.0, len(ratings)))
    for cor, rating in zip(cores, ratings):
        d = tabela_a[tabela_a["rating"] == rating].sort_values("safra")
        valores = d["observado"].clip(lower=1e-4) * 100
        eixo.plot([posicao[s] for s in d["safra"]], valores, color=cor, lw=1.6, marker="o", markersize=3)
        eixo.text(len(safras) - 0.4, valores.iloc[-1], f" {rating}", va="center", fontsize=8, color="#52514e")
    eixo.set_yscale("log")
    eixo.set_xticks(range(len(safras)))
    eixo.set_xticklabels([_rotulo_safra(s) for s in safras], rotation=45, ha="right", fontsize=7)
    eixo.set_xlim(-0.5, len(safras) - 0.5)
    eixo.set_ylabel(("PD" if cfg["tarefa"] == "pd" else "LGD") + " observada (%), escala log")
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(axis="y", color="#e4e3df", lw=0.8, which="both")
    eixo.set_title("Ordem dos ratings por safra (linhas que se cruzam = inversão)", loc="left", fontsize=11, pad=20)
    figura.tight_layout()
    return figura


def grafico_mapa_status(tabela_a):
    """Mapa rating × safra colorido pelo status."""
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch
    mapa = {"ok": 0, "atencao": 1, "alerta": 2}
    grade = tabela_a.pivot_table(index="rating", columns="safra", values="status",
                                 aggfunc="first").sort_index(ascending=False)
    valores = grade.apply(lambda c: c.map(mapa)).to_numpy(dtype=float)
    figura, eixo = plt.subplots(figsize=(max(8, 0.35 * grade.shape[1] + 2), 0.38 * grade.shape[0] + 1.6), dpi=130)
    eixo.imshow(np.ma.masked_invalid(valores), aspect="auto",
                cmap=ListedColormap([CORES_STATUS["ok"], CORES_STATUS["atencao"], CORES_STATUS["alerta"]]),
                vmin=0, vmax=2)
    eixo.set_yticks(range(grade.shape[0]))
    eixo.set_yticklabels(grade.index, fontsize=8)
    eixo.set_xticks(range(grade.shape[1]))
    eixo.set_xticklabels([_rotulo_safra(s) for s in grade.columns], rotation=45, ha="right", fontsize=7)
    eixo.legend(handles=[Patch(color=CORES_STATUS[s], label=NOME_STATUS[s]) for s in mapa],
                loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False, fontsize=8)
    eixo.set_title("Status do backtest: rating × safra", loc="left", fontsize=11)
    figura.tight_layout()
    return figura


def grafico_ponte(tabela_d, cfg):
    """Ponte da carteira por período: DES → mix → composição → calibração → observado."""
    import matplotlib.pyplot as plt
    linhas = tabela_d.set_index("particao")
    particoes = [p for p in ("treino", "teste", "oot") if p in linhas.index]
    figura, eixos = plt.subplots(1, len(particoes), figsize=(4.2 * len(particoes), 4), dpi=130, sharey=True)
    eixos = np.atleast_1d(eixos)
    for eixo, particao in zip(eixos, particoes):
        l = linhas.loc[particao]
        degraus = [("DES", l["valor_des"], None), ("mix", l["efeito_mix"], "d"),
                   ("composição", l["gap_composicao"], "d"), ("calibração", l["gap_calibracao"], "d"),
                   ("observado", l["observado"], None)]
        acumulado = 0.0
        for i, (rotulo, valor, tipo) in enumerate(degraus):
            if tipo is None:                       # barra cheia: DES ou observado
                eixo.bar(i, valor * 100, color="#2a78d6", width=0.6)
                acumulado, topo, texto = valor, valor, f"{valor:.2%}"
            else:                                  # degrau: sobe (vermelho) ou desce (verde)
                eixo.bar(i, valor * 100, bottom=acumulado * 100, width=0.6,
                         color="#d03b3b" if valor > 0 else "#0ca30c")
                topo = max(acumulado, acumulado + valor)
                acumulado += valor
                texto = f"{valor:+.2%}"
            eixo.text(i, topo * 100, texto.replace(".", ","), ha="center", va="bottom", fontsize=7)
        eixo.set_xticks(range(len(degraus)))
        eixo.set_xticklabels([d[0] for d in degraus], fontsize=8, rotation=20)
        eixo.set_title(particao.upper(), fontsize=10)
        eixo.spines[["top", "right"]].set_visible(False)
        eixo.grid(axis="y", color="#e4e3df", lw=0.8)
        eixo.set_axisbelow(True)
    eixos[0].set_ylabel(("PD" if cfg["tarefa"] == "pd" else "LGD") + " (%)")
    figura.suptitle("Ponte da carteira: do DES ao observado", x=0.01, ha="left", fontsize=11)
    figura.tight_layout()
    return figura

## 8b. Painel de estabilidade do modelo

Uma figura com quatro gráficos, para ler a estabilidade do modelo ao longo do tempo. Os três primeiros usam **todas as safras do DES e do OOT** no mesmo eixo, com linhas verticais pontilhadas separando treino, teste e OOT:

| gráfico | o que mostra | como ler |
|---|---|---|
| 1. taxa por rating e safra | taxa de maus (PD) ou LGD média observada de cada rating, em escala logarítmica | linhas paralelas e na ordem = ratings estáveis; linhas que se cruzam = inversão naquela safra |
| 2. volumetria e PSI | barras empilhadas: participação de cada rating em cada safra; linha: PSI da distribuição de ratings da safra contra o DES | barras com a mesma composição = população estável; PSI < 0,10 estável, 0,10 a 0,25 atenção, > 0,25 mudança relevante |
| 3. discriminação por safra | PD: KS de cada safra; LGD: Spearman entre o score e a LGD. Linha tracejada: o valor no DES inteiro. Barras ao fundo: número de maus (PD) ou de contratos (LGD) | queda persistente no OOT = perda de poder; oscilação em safras com poucos maus é ruído |
| 4. ordenação | taxa de cada rating no DES (a régua), no teste e no OOT | as três curvas devem crescer de A ao último rating; uma curva que desce indica inversão no período |

**PSI dos ratings, exemplo.** Rating A tem 10% do DES e 6% de uma safra do OOT: (0,06 − 0,10) · ln(0,06 / 0,10) = (−0,04) · (−0,511) = 0,020. Somando a mesma conta em todos os ratings, sai o PSI da safra.

In [ ]:
from pyspark.sql import Window

N_FAIXAS_KS = 10_000


def _ks_de_histograma(h):
    """KS a partir de contagens por faixa fina do score (colunas faixa, n, eventos)."""
    h = h.sort_values("faixa")
    maus, bons = h["eventos"], h["n"] - h["eventos"]
    if maus.sum() == 0 or bons.sum() == 0:
        return np.nan
    return float((maus.cumsum() / maus.sum() - bons.cumsum() / bons.sum()).abs().max())


def discriminacao_por_safra(base_com_rating, cfg, coluna_score="score_calibrado"):
    """
    Discriminação de cada safra (DES e OOT) e do DES inteiro.

    PD: KS, calculado de um histograma do score em 10.000 faixas finas por safra
        (uma agregação Spark, sem ordenar a base).
    LGD: correlação de Spearman entre o score e a LGD em cada safra (ranking com
        empates recebendo a posição média, por janelas Spark).

    Devolve (tabela por safra, valor no DES).
    """
    colunas = cfg["colunas"]
    y = F.col(colunas["alvo"]).cast("double")
    base = (base_com_rating.filter(F.col("particao").isin("treino", "teste", "oot"))
            .select(F.col(colunas["safra"]).alias("safra"), "particao", y.alias("_y"),
                    F.col(coluna_score).cast("double").alias("_score")))
    if cfg["tarefa"] == "pd":
        faixa = F.least(F.floor(F.col("_score") * N_FAIXAS_KS).cast("int"), F.lit(N_FAIXAS_KS - 1))
        h = (base.groupBy("safra", "particao", faixa.alias("faixa"))
                 .agg(F.count(F.lit(1)).alias("n"), F.sum("_y").alias("eventos")).toPandas())
        linhas = [{"safra": safra, "particao": particao, "n": int(g["n"].sum()),
                   "eventos": int(g["eventos"].sum()), "discriminacao": _ks_de_histograma(g)}
                  for (safra, particao), g in h.groupby(["safra", "particao"])]
        des = h[h["particao"].isin(PARTICOES_DES)].groupby("faixa", as_index=False)[["n", "eventos"]].sum()
        return pd.DataFrame(linhas).sort_values("safra").reset_index(drop=True), _ks_de_histograma(des)

    def com_ranking(sdf, chave):
        """Ranking médio (empates com a posição média) de _score e _y dentro de `chave`."""
        for coluna in ("_score", "_y"):
            posicao = F.row_number().over(Window.partitionBy(*chave).orderBy(coluna))
            sdf = (sdf.withColumn(f"_pos{coluna}", posicao)
                      .withColumn(f"_rank{coluna}", F.avg(f"_pos{coluna}").over(Window.partitionBy(*chave, coluna))))
        return sdf

    por_safra = (com_ranking(base, ["safra"])
                 .groupBy("safra", "particao")
                 .agg(F.count(F.lit(1)).alias("n"), F.count(F.lit(1)).alias("eventos"),
                      F.corr("_rank_score", "_rank_y").alias("discriminacao")).toPandas())
    des = (com_ranking(base.filter(F.col("particao").isin(*PARTICOES_DES)).withColumn("_grupo", F.lit(1)), ["_grupo"])
           .agg(F.corr("_rank_score", "_rank_y")).first()[0])
    return por_safra.sort_values("safra").reset_index(drop=True), float(des) if des is not None else np.nan


def psi_ratings_por_safra(tabela_a, regua):
    """PSI da distribuição de ratings de cada safra contra a distribuição do DES (régua)."""
    esperado = regua.set_index("rating")["participacao_des"]
    linhas = []
    for safra, grupo in tabela_a.groupby("safra"):
        atual = grupo.set_index("rating")["n"] / grupo["n"].sum()
        e = esperado.reindex(esperado.index.union(atual.index), fill_value=0.0)
        a = atual.reindex(e.index, fill_value=0.0)
        linhas.append({"safra": safra, "psi_ratings": float(((a - e) * np.log((a + 1e-6) / (e + 1e-6))).sum())})
    return pd.DataFrame(linhas)


def _linhas_periodos(eixo, particoes, rotular=False):
    """Linhas verticais pontilhadas entre treino, teste e OOT (e os nomes no topo, se pedido)."""
    inicio = 0
    for i in range(1, len(particoes) + 1):
        if i == len(particoes) or particoes[i] != particoes[inicio]:
            if rotular:
                eixo.text((inicio + i - 1) / 2, 1.03, f"{particoes[inicio].upper()} ({i - inicio})",
                          transform=eixo.get_xaxis_transform(), ha="center", fontsize=8,
                          color="#52514e", fontweight="bold")
            if inicio > 0:
                eixo.axvline(inicio - 0.5, color="#52514e", lw=1.1, ls=(0, (2, 2)), zorder=1)
            inicio = i


def grafico_estabilidade(tabela_a, tabela_b, regua, estabilidade, discriminacao_des, cfg):
    """Painel de estabilidade: taxa por rating, volumetria com PSI, discriminação e ordenação."""
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    pd_ou_lgd = "PD" if cfg["tarefa"] == "pd" else "LGD"
    safras = sorted(tabela_a["safra"].unique())
    posicao = {s: i for i, s in enumerate(safras)}
    particoes = tabela_a.drop_duplicates("safra").set_index("safra").loc[safras, "particao"].tolist()
    ratings = list(regua["rating"])
    cores = dict(zip(ratings, plt.cm.Blues(np.linspace(0.3, 1.0, len(ratings)))))
    x = np.arange(len(safras))
    rotulos_x = [_rotulo_safra(s) for s in safras]

    figura = plt.figure(figsize=(12, 15), dpi=120)
    grade = figura.add_gridspec(4, 1, height_ratios=[1.2, 1, 1, 0.9], hspace=0.55)
    eixos = [figura.add_subplot(grade[i]) for i in range(4)]

    # 1. taxa por rating e safra
    eixo = eixos[0]
    for rating in ratings:
        d = tabela_a[tabela_a["rating"] == rating].sort_values("safra")
        # Taxa zero não existe em escala log: a safra fica sem ponto (lacuna na linha).
        valores = d["observado"].where(d["observado"] > 0) * 100
        eixo.plot([posicao[s] for s in d["safra"]], valores, color=cores[rating], lw=1.6, marker="o", markersize=3)
        ultimo = valores.dropna()
        if len(ultimo):
            eixo.text(len(safras) - 0.4, ultimo.iloc[-1], f" {rating}", va="center", fontsize=8, color="#52514e")
    eixo.set_yscale("log")
    eixo.set_ylabel(f"{pd_ou_lgd} observada (%), log")
    eixo.set_title(f"1. {pd_ou_lgd} observada por rating e safra (taxa zero = lacuna na linha)",
                   loc="left", fontsize=11, pad=22)
    _linhas_periodos(eixo, particoes, rotular=True)

    # 2. volumetria por rating e PSI
    eixo = eixos[1]
    contagem = (tabela_a.pivot_table(index="safra", columns="rating", values="n", aggfunc="sum")
                        .reindex(index=safras, columns=ratings).fillna(0))
    participacao = contagem.div(contagem.sum(axis=1), axis=0) * 100
    base_barra = np.zeros(len(safras))
    for rating in ratings:
        eixo.bar(x, participacao[rating], bottom=base_barra, color=cores[rating], width=0.8,
                 edgecolor="white", linewidth=0.4, label=rating)
        base_barra += participacao[rating].to_numpy()
    eixo.set_ylim(0, 100)
    eixo.set_ylabel("participação na safra (%)")
    eixo_psi = eixo.twinx()
    psi = estabilidade.set_index("safra").reindex(safras)["psi_ratings"]
    eixo_psi.plot(x, psi, color="#eb6834", lw=2, marker="o", markersize=4, zorder=5)
    for limite in (0.10, 0.25):
        eixo_psi.axhline(limite, color="#eb6834", lw=0.9, ls=(0, (5, 3)), alpha=0.7)
    eixo_psi.set_ylim(0, max(0.3, float(np.nanmax(psi.to_numpy())) * 1.15 if psi.notna().any() else 0.3))
    eixo_psi.set_ylabel("PSI dos ratings vs DES", color="#eb6834")
    eixo_psi.spines[["top"]].set_visible(False)
    eixo.legend(handles=[*[Line2D([], [], color=cores[r], lw=6, label=r) for r in ratings],
                         Line2D([], [], color="#eb6834", lw=2, marker="o", label="PSI"),
                         Line2D([], [], color="#eb6834", lw=0.9, ls=(0, (5, 3)), label="PSI 0,10 e 0,25")],
                loc="upper left", bbox_to_anchor=(1.1, 1.0), frameon=False, fontsize=8)
    eixo.set_title("2. Volumetria por rating e PSI da distribuição de ratings", loc="left", fontsize=11, pad=8)
    _linhas_periodos(eixo, particoes)

    # 3. discriminação por safra
    eixo = eixos[2]
    disc = estabilidade.set_index("safra").reindex(safras)
    eixo_volume = eixo.twinx()
    eixo_volume.bar(x, disc["eventos"], color="#e4e3df", width=0.8, zorder=0)
    eixo_volume.set_ylabel("maus na safra" if cfg["tarefa"] == "pd" else "contratos na safra", color="#8a8984")
    eixo_volume.spines[["top"]].set_visible(False)
    eixo.set_zorder(eixo_volume.get_zorder() + 1)
    eixo.patch.set_visible(False)
    nome_metrica = "KS" if cfg["tarefa"] == "pd" else "Spearman (score × LGD)"
    eixo.plot(x, disc["discriminacao"], color="#2a78d6", lw=2, marker="o", markersize=4)
    eixo.axhline(discriminacao_des, color="#2a78d6", lw=1, ls=(0, (5, 3)))
    eixo.text(-0.4, discriminacao_des, f"DES {discriminacao_des:.3f}".replace(".", ","),
              va="bottom", fontsize=8, color="#2a78d6")
    eixo.set_ylabel(nome_metrica)
    eixo.set_title(f"3. {nome_metrica} por safra (tracejado: DES inteiro)", loc="left", fontsize=11, pad=8)
    _linhas_periodos(eixo, particoes)

    for eixo in eixos[:3]:
        eixo.set_xticks(x)
        eixo.set_xticklabels(rotulos_x, rotation=45, ha="right", fontsize=7)
        eixo.set_xlim(-0.5, len(safras) - 0.5)
        eixo.grid(axis="y", color="#e4e3df", lw=0.8, which="major")
        eixo.spines[["top", "right"]].set_visible(False)

    # 4. ordenação: DES (régua), teste e OOT
    eixo = eixos[3]
    xr = np.arange(len(ratings))
    eixo.plot(xr, regua["valor_regua"] * 100, color="#52514e", lw=2, marker="o", label="DES (régua)")
    estilos = {"teste": ("#2a78d6", (0, (5, 3))), "oot": ("#eb6834", "-")}
    for particao, (cor, estilo) in estilos.items():
        d = tabela_b[tabela_b["particao"] == particao].set_index("rating").reindex(ratings)
        eixo.plot(xr, d["observado"] * 100, color=cor, lw=2, ls=estilo, marker="o", label=particao.upper())
    eixo.set_xticks(xr)
    eixo.set_xticklabels(ratings)
    eixo.set_xlabel("rating")
    eixo.set_ylabel(f"{pd_ou_lgd} observada (%)")
    eixo.grid(axis="y", color="#e4e3df", lw=0.8)
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.legend(loc="upper left", frameon=False, fontsize=8)
    eixo.set_title("4. Ordenação: taxa por rating no DES, no teste e no OOT", loc="left", fontsize=11, pad=8)

    figura.suptitle("Estabilidade do modelo", x=0.06, y=0.995, ha="left", fontsize=13, fontweight="bold")
    figura.subplots_adjust(top=0.94)
    return figura

## 9. Execução das etapas 7 e 7b

In [ ]:
def executa_etapa_regua(spark, base_calibrada, cfg, gravar=True):
    """
    Etapas 7 (régua) e 7b (backtest).

    Devolve
        regua               pandas, uma linha por rating
        registro_fusoes     pandas, uma linha por fusão
        base_com_rating     Spark, com `rating` e `valor_regua` (todas as partições)
        backtest            {rating_safra, rating_periodo, total_safra, total_periodo, ordem_por_safra}
        figura_fusao, figuras_rating ({rating: figura}), figura_ordem, figura_mapa_status, figura_ponte
        estabilidade        pandas por safra: n, eventos, KS (PD) ou Spearman (LGD), PSI dos ratings
        figura_estabilidade painel de estabilidade (8b)
    """
    cortes_finos, tabela_finas = faixas_finas(base_calibrada, cfg)
    n_inicial, grupos_iniciais = escolhe_faixas(tabela_finas, cfg)
    grupos, registro = funde_faixas(grupos_iniciais, tabela_finas, cfg)
    regua = monta_regua(grupos, cortes_finos, tabela_finas, cfg)
    print(f"[régua] {n_inicial} faixas iniciais → {len(regua)} ratings; "
          f"{len(registro)} fusões ({int(registro['significativa'].sum()) if len(registro) else 0} "
          f"por inversão/diferença significativa)")
    exibe(regua, "Régua")
    if len(registro):
        exibe(registro, "Registro de fusões")
    significativas = registro[(registro["motivo"] == "inversão no teste") & registro["significativa"]] if len(registro) else registro
    if len(significativas):
        print(f"[régua] ACHADO: {len(significativas)} inversão(ões) significativa(s) no teste — o modelo "
              f"não ordena bem nessa região do score.")

    base_com_rating = guarda_resultado(spark, aplica_regua(base_calibrada, regua), "regua_base_com_rating", cfg)
    backtest = tabelas_backtest(agrega_backtest(base_com_rating, cfg), regua, cfg)
    exibe(backtest["rating_periodo"][["rating", "particao", "n", "observado", "regua", "modelo_medio",
                                      "ic_inf", "ic_sup", "desvio_relativo", "status"]], "Backtest: rating × período")
    exibe(backtest["total_periodo"][["particao", "n", "valor_des", "efeito_mix", "gap_composicao",
                                     "gap_calibracao", "observado", "status"]], "Backtest: ponte da carteira")
    recalibrar = sorted(backtest["rating_safra"].loc[backtest["rating_safra"]["recalibrar"], "rating"].unique())
    if recalibrar:
        print(f"[backtest] ratings com {cfg['backtest_regua']['n_safras_persistencia']} ou mais alertas seguidos "
              f"(recalibrar): {recalibrar}")

    figura_fusao = grafico_fusao(n_inicial, grupos_iniciais, regua, tabela_finas, cfg)
    figuras_rating = {r: grafico_rating_safra(backtest["rating_safra"], r, cfg) for r in regua["rating"]}
    figura_ordem = grafico_ordem_por_safra(backtest["rating_safra"], cfg)
    figura_mapa = grafico_mapa_status(backtest["rating_safra"])
    figura_ponte = grafico_ponte(backtest["total_periodo"], cfg)

    # Painel de estabilidade (8b): discriminação e PSI dos ratings por safra.
    discriminacao, discriminacao_des = discriminacao_por_safra(base_com_rating, cfg)
    estabilidade = discriminacao.merge(psi_ratings_por_safra(backtest["rating_safra"], regua), on="safra", how="left")
    estabilidade = estabilidade.rename(columns={"discriminacao": "ks" if cfg["tarefa"] == "pd" else "spearman"})
    exibe(estabilidade, "Estabilidade por safra")
    figura_estabilidade = grafico_estabilidade(
        backtest["rating_safra"], backtest["rating_periodo"], regua,
        estabilidade.rename(columns={"ks": "discriminacao", "spearman": "discriminacao"}), discriminacao_des, cfg)
    registra_figura_mlflow(figura_estabilidade, "regua/estabilidade.png")
    registra_figura_mlflow(figura_fusao, "regua/fusao.png")
    for rating, figura in figuras_rating.items():
        registra_figura_mlflow(figura, f"regua/backtest_rating_{rating}.png")
    registra_figura_mlflow(figura_ordem, "regua/ordem_por_safra.png")
    registra_figura_mlflow(figura_mapa, "regua/mapa_status.png")
    registra_figura_mlflow(figura_ponte, "regua/ponte.png")

    if gravar:
        grava_tabela_delta(spark, regua, "regua", cfg)
        if len(registro):
            grava_tabela_delta(spark, registro, "regua_fusoes", cfg)
        for nome, tabela in backtest.items():
            grava_tabela_delta(spark, tabela, f"backtest_{nome}", cfg)
        grava_tabela_delta(spark, estabilidade, "estabilidade_safra", cfg)

    return {"regua": regua, "registro_fusoes": registro, "base_com_rating": base_com_rating,
            "backtest": backtest, "n_faixas_iniciais": n_inicial, "figura_fusao": figura_fusao,
            "figuras_rating": figuras_rating, "figura_ordem": figura_ordem,
            "figura_mapa_status": figura_mapa, "figura_ponte": figura_ponte,
            "estabilidade": estabilidade, "figura_estabilidade": figura_estabilidade}

## 10. Exemplo de uso

```python
etapa7 = executa_etapa_regua(spark, etapa56["base_calibrada"], cfg)
etapa7["regua"]                                  # ratings, cortes e PD de cada um
etapa7["backtest"]["rating_safra"]               # tabela A
etapa7["figuras_rating"]["H"]                    # gráfico do rating H
etapa7["base_com_rating"]                        # segue para a PD lifetime (notebook 06)
```